In [1]:
import os

dataset_file_names = list(filter(lambda p: ".jsonl" in p, next(os.walk("."))[2]))

In [2]:
import json

records = []
for path in dataset_file_names:
    with open(path) as file:
        records.extend([json.loads(line) for line in file])
len(records)

182071

In [3]:
def hdict(record: dict):
    move_tuple = (
        tuple(record["move"]["src"]) + tuple(record["move"]["dst"])
        if record["move"] is not None
        else ()
    )
    return (
        tuple(tuple(row) for row in record["board"])
        + move_tuple
        + tuple(record["place"]["spot"])
    )


len({hdict(record) for record in records})

124263

In [4]:
import torch


def augment_record(record: dict) -> list[dict]:
    rot_records = []
    board = torch.tensor(record["board"])

    def _naive_rot_pos(pos: list[int], k: int) -> list[int]:
        temp_board = torch.zeros_like(board)
        temp_board[*pos] = 1
        row, col = (temp_board.rot90(k) == 1).nonzero(as_tuple=True)
        return [row[0].item(), col[0].item()]  # type: ignore

    for k in range(1, 4):
        rot_record = {
            "board": board.rot90(k).detach().tolist(),
            "move": (
                None
                if record["move"] is None
                else {
                    "src": _naive_rot_pos(record["move"]["src"], k),
                    "dst": _naive_rot_pos(record["move"]["dst"], k),
                }
            ),
            "place": {"spot": _naive_rot_pos(record["place"]["spot"], k)},
        }
        rot_records.append(rot_record)

    return rot_records


records[0], augment_record(records[0])

({'board': [[-1, 1, -1, 0], [-1, 0, 1, 0], [1, 1, -1, 0], [1, -1, 0, 0]],
  'move': None,
  'place': {'spot': [3, 2]}},
 [{'board': [[0, 0, 0, 0], [-1, 1, -1, 0], [1, 0, 1, -1], [-1, -1, 1, 1]],
   'move': None,
   'place': {'spot': [1, 3]}},
  {'board': [[0, 0, -1, 1], [0, -1, 1, 1], [0, 1, 0, -1], [0, -1, 1, -1]],
   'move': None,
   'place': {'spot': [0, 1]}},
  {'board': [[1, 1, -1, -1], [-1, 1, 0, 1], [0, -1, 1, -1], [0, 0, 0, 0]],
   'move': None,
   'place': {'spot': [2, 0]}}])

In [6]:
def naive_rot_pos(pos: list[int], k: int) -> list[int]:
    temp_board = torch.zeros(4, 4)
    temp_board[*pos] = 1
    row, col = (temp_board.rot90(k) == 1).nonzero(as_tuple=True)
    return [row[0].item(), col[0].item()]  # type: ignore

In [7]:
all_dataset = []
for record in records:
    all_dataset.append(record)
    all_dataset.extend(augment_record(record))

In [8]:
import json
from dataclasses import asdict, dataclass


@dataclass(frozen=True)
class DatasetEntry:
    board: list[list[int]]
    move: dict | None
    place: dict


def save_dataset_jsonl(entries: list[DatasetEntry], path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        f.writelines(json.dumps(asdict(entry)) + "\n" for entry in entries)

In [9]:
all_dataset[0]

{'board': [[-1, 1, -1, 0], [-1, 0, 1, 0], [1, 1, -1, 0], [1, -1, 0, 0]],
 'move': None,
 'place': {'spot': [3, 2]}}

In [10]:
def to_entry(data: dict) -> DatasetEntry:
    return DatasetEntry(
        board=data["board"],
        move=data["move"],
        place=data["place"],
    )

In [11]:
to_save_dataset = [to_entry(data) for data in all_dataset]
save_dataset_jsonl(to_save_dataset, "orbito_dataset_whole.jsonl")

In [12]:
len(to_save_dataset)

728284